# Goal and Work Summary

**Goal**: Train and evaluate DeepLabV3, Unet, or UnetPlusPlus on native-resolution tiles of green rough oak planks (classes come from the prepared dataset, e.g. Knot, Rot, Sapwood, Stain). Boards are tiled before resizing so small defects survive the network input transform. Ordinary test metrics in this notebook are **per-tile**; an optional slow cell also reports stitched board-level IoU/F1.

## Workflow Stages

**Stage 1 - Data audit and class normalization**
- Scan raw line-scan data and normalize class naming with reusable helpers in `notebooks/functions`.

**Stage 2 - Class filtering and split preparation**
- Filter low-frequency classes and build reproducible board-level train/val/test splits.

**Stage 3 - Native-resolution tile generation**
- Crop each organized board into configurable native-resolution windows.
- Crop images and indexed masks with identical NumPy boxes.
- Keep every validation/test tile; optionally drop background-only training tiles.

**Stage 4 - Tile resizing and augmentation**
- Resize each tile to `TILE_RESIZE_HEIGHT` x `TILE_RESIZE_WIDTH` for the network.
- Preserve the existing augmentation and mask-ID remapping behavior.

**Stage 5 - Optuna and final training**
- Tune and train the existing DeepLabV3 workflow with isolated tiled-study artifacts.

**Stage 6 - Evaluation and export**
- Report required per-tile metrics, optionally stitch native tile predictions for board-level metrics, and export an EBI-compatible tile-input ONNX model.
- The exported model is compatible with the existing Tiled / Crop-Based Inference section of `notebooks/app.py`; use matching tile width and height values.

## References

| Library | Purpose |
| --- | --- |
| PyTorch | Training and inference |
| segmentation_models_pytorch | DeepLabV3 architecture |
| Optuna | Hyperparameter optimization |
| Albumentations | Augmentation and normalization |
| scikit-learn | Evaluation metrics |
| ONNX Runtime | Export verification |

In [ ]:
# Prepare dataset + core imports
from functions.data_pipeline import prepare_dataset
import segmentation_models_pytorch as smp
# Launch TensorBoard Session
from torch.utils.tensorboard import SummaryWriter
import optuna
from functions.oak_hf_dataset import ensure_oak_raw_data

# Change this to use a custom folder name under data/
RAW_FOLDER_NAME = "26-9-30 Merge"
RAW_DIR = ensure_oak_raw_data(raw_folder_name=RAW_FOLDER_NAME)

data_root, class_mapping = prepare_dataset(
    raw_data_folder=str(RAW_DIR),
    mask_threshold=10,
    val_split=0.16,
    test_split=0.2,
)

In [ ]:
# Native-resolution tiling stage
from pathlib import Path
import json
import numpy as np
from PIL import Image

from functions.tile_dataset import create_tiled_dataset, tessellate_boxes
import matplotlib.pyplot as plt

TILE_MAX_X = 1088
TILE_MAX_Y = 1088
TILE_STEP_X = None
TILE_STEP_Y = None
TILE_EDGE_MODE = "flush"
DROP_BACKGROUND_TRAIN_TILES = False
MIN_TILE_DEFECT_PIXELS = 0
TILED_DATA_DIR = f"../data/tiles/x{TILE_MAX_X}_y{TILE_MAX_Y}_sx{TILE_STEP_X or TILE_MAX_X}_sy{TILE_STEP_Y or TILE_MAX_Y}_{TILE_EDGE_MODE}"

tiled_data_root, tile_stats = create_tiled_dataset(
    data_root,
    TILED_DATA_DIR,
    TILE_MAX_X,
    TILE_MAX_Y,
    step_x=TILE_STEP_X,
    step_y=TILE_STEP_Y,
    edge_mode=TILE_EDGE_MODE,
    drop_background_train_tiles=DROP_BACKGROUND_TRAIN_TILES,
    min_tile_defect_pixels=MIN_TILE_DEFECT_PIXELS,
)
print("Tiled dataset stats:", tile_stats)

with open(Path(data_root) / "split.json", encoding="utf-8") as handle:
    source_split = json.load(handle)
sample_board = next(iter(source_split["train"]), None)
if sample_board:
    sample_image = np.asarray(Image.open(Path(data_root) / "images" / f"{sample_board}_Col.tif").convert("RGB"))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.imshow(sample_image)
    for ymin, ymax, xmin, xmax in tessellate_boxes(
        *sample_image.shape[:2], TILE_MAX_X, TILE_MAX_Y, TILE_STEP_X, TILE_STEP_Y, TILE_EDGE_MODE
    ):
        ax.plot(
            [xmin, xmax, xmax, xmin, xmin],
            [ymin, ymin, ymax, ymax, ymin],
            color="yellow",
            linewidth=1,
        )
    ax.set_title(f"Native tile boxes: {sample_board}")
    ax.axis("off")
    plt.show()

In [ ]:
# Select the segmentation architecture used by training, evaluation, and export.
# Supported values: "DeepLabV3", "Unet", "UnetPlusPlus".
import torch
from datetime import datetime
from pathlib import Path

GPU_ID = 0
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE = torch.device(f"cuda:{GPU_ID}")
    print(f"Using GPU {GPU_ID}: {torch.cuda.get_device_name(GPU_ID)}")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")

MODEL_ARCH      = "Unet"
ENCODER_NAME    = "resnet34"
TUNE_ENCODER    = False  # True: Optuna compares encoders; False: use ENCODER_NAME
ENCODER_WEIGHTS = "imagenet"

DATA_ROOT       = tiled_data_root
TILE_RESIZE_HEIGHT = 1088
TILE_RESIZE_WIDTH  = 1088
BATCH_SIZE      = 8
NUM_WORKERS     = 4
OPTUNA_BATCH_SIZE = 2  # Keep trial host-memory usage below the TIFF decode peak.
EVALUATION_BATCH_SIZE = 1
NUM_CLASSES     = len(class_mapping) + 1

# Remap disk mask class ids to training ids (leave {} for no remap).
MASK_ID_REMAP   = {}

TRANSFER_LEARNING = False
TRANSFER_CHECKPOINT_PATH = "../models/CC_DeepLabV3_FullImage_Optuna/model_best.pt"
FREEZE_ENCODER_EPOCHS = 5
UNFREEZE_AFTER_FREEZE = True
TRANSFER_LR_FROZEN = 3e-4
TRANSFER_LR_UNFROZEN = 1e-4

N_TRIALS        = 5
TRIAL_EPOCHS    = 30
ENCODER_MODE    = "search" if TUNE_ENCODER else f"fixed_{ENCODER_NAME}"
OPTUNA_DB_PATH  = "../logs/optuna/TILED_Optuna.db"
STUDY_NAME = f"{MODEL_ARCH}_tiled_optuna_{ENCODER_MODE}_{datetime.now():%Y%m%d_%H%M%S}"
RESUME_PREVIOUS_STUDY = True
RESUME_STUDY_NAME = None

LEARNING_RATE   = 1e-3
NUM_EPOCHS      = 150
EARLY_STOP_PATIENCE = 15
CHECKPOINT_NAME = f"TILE_{MODEL_ARCH}_Tiled_Optuna_{datetime.now():%Y%m%d_%H%M%S}"
LOG_NAME        = f"TILE_{MODEL_ARCH}_Tiled_Optuna_{datetime.now():%Y%m%d_%H%M%S}"
ONNX_NAME       = f"tile_{MODEL_ARCH}_tiled_optuna_{datetime.now():%Y%m%d_%H%M%S}.onnx"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models" / CHECKPOINT_NAME
METRICS_RUN_NAME = f"tile_{MODEL_ARCH}_tiled_optuna_{datetime.now():%Y%m%d_%H%M%S}"
METRICS_DIR = PROJECT_ROOT / "data" / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import json
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


def apply_mask_id_remap(mask):
    """Map on-disk combined-mask class ids to training ids."""
    remap = MASK_ID_REMAP if MASK_ID_REMAP else {}
    if not remap:
        return mask
    if isinstance(mask, torch.Tensor):
        t = mask.long()
        device = t.device
        mv = int(t.max().item()) if t.numel() else 0
        hi = max(mv, max(remap.keys()), max(remap.values()))
        lut = torch.arange(hi + 1, dtype=torch.long, device=device)
        for src, dst in remap.items():
            src, dst = int(src), int(dst)
            if 0 <= src <= hi:
                lut[src] = dst
        return lut[t].to(mask.dtype)

    arr = np.asarray(mask)
    mv = int(arr.max()) if arr.size > 0 else 0
    hi = max(mv, max(remap.keys()), max(remap.values()))
    lut = np.arange(hi + 1, dtype=np.int64)
    for src, dst in remap.items():
        src, dst = int(src), int(dst)
        if 0 <= src <= hi:
            lut[src] = dst
    return lut[arr.astype(np.int64)].astype(arr.dtype, copy=False)


_remap_dbg = MASK_ID_REMAP if MASK_ID_REMAP else {}
print("Training class structure (after MASK_ID_REMAP):")
print("  0: Background")
for _cls_name, _disk_id in sorted(class_mapping.items(), key=lambda kv: kv[1]):
    _train_id = int(_remap_dbg.get(_disk_id, _disk_id))
    print(f"  {_train_id}: {_cls_name}    (on-disk id {_disk_id})")


class TiledSegmentationDataset(Dataset):
    def __init__(self, data_root, split="train", transform=None):
        self.data_root = Path(data_root)
        self.split = split
        self.transform = transform
        with open(self.data_root / "split.json", "r") as f:
            split_info = json.load(f)
        self.image_ids = split_info[split]
        self.images_dir = self.data_root / "images"
        self.masks_dir = self.data_root / "combined_masks"
        print(f"Loaded {split} dataset: {len(self.image_ids)} tiles")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        tile_id = self.image_ids[idx]
        with Image.open(self.images_dir / f"{tile_id}_Col.tif") as image_file:
            image = np.asarray(image_file.convert("RGB")).copy()
        with Image.open(self.masks_dir / f"{tile_id}_mask.png") as mask_file:
            mask = np.asarray(mask_file).copy()
        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]
        return image, apply_mask_id_remap(mask).long()


def get_training_augmentation_full(image_height=TILE_RESIZE_HEIGHT, image_width=TILE_RESIZE_WIDTH):
    aug_list = [
        A.Resize(image_height, image_width),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
        A.OneOf([A.RandomBrightnessContrast(p=1), A.RandomGamma(p=1)], p=0.3),
        A.OneOf([A.GaussNoise(p=1), A.GaussianBlur(p=1)], p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ]
    if image_height == image_width:
        aug_list.insert(3, A.RandomRotate90(p=0.5))
    return A.Compose(aug_list)


def get_validation_augmentation_full(image_height=TILE_RESIZE_HEIGHT, image_width=TILE_RESIZE_WIDTH):
    return A.Compose([
        A.Resize(image_height, image_width),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def create_dataloaders_full(
    data_root,
    batch_size=BATCH_SIZE,
    num_workers=4,
    image_height=TILE_RESIZE_HEIGHT,
    image_width=TILE_RESIZE_WIDTH,
    pin_memory=True,
):
    train_dataset = TiledSegmentationDataset(
        data_root=data_root,
        split="train",
        transform=get_training_augmentation_full(image_height=image_height, image_width=image_width),
    )
    val_dataset = TiledSegmentationDataset(
        data_root=data_root,
        split="val",
        transform=get_validation_augmentation_full(image_height=image_height, image_width=image_width),
    )
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers,
        pin_memory=pin_memory, drop_last=True,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return train_loader, val_loader

In [ ]:
# Runtime setup (DEVICE set in Configuration cell above)

inv_train = {}
for name, disk_id in class_mapping.items():
    train_id = int(MASK_ID_REMAP.get(disk_id, disk_id)) if MASK_ID_REMAP else int(disk_id)
    if train_id in inv_train:
        raise ValueError(
            f"MASK_ID_REMAP collision: train_id={train_id} maps to both "
            f"{inv_train[train_id]} and {name}"
        )
    inv_train[train_id] = name

missing_train_ids = [i for i in range(1, NUM_CLASSES) if i not in inv_train]
if missing_train_ids:
    raise ValueError(f"Missing class names for training ids {missing_train_ids}.")

CLASS_NAMES = ["Background"] + [inv_train[i] for i in range(1, NUM_CLASSES)]
assert len(CLASS_NAMES) == NUM_CLASSES

# Compatibility names keep the copied training utilities unchanged.
IMAGE_HEIGHT = TILE_RESIZE_HEIGHT
IMAGE_WIDTH = TILE_RESIZE_WIDTH

print("CLASS_NAMES by training id:", {i: n for i, n in enumerate(CLASS_NAMES)})
print(f"Model architecture: {MODEL_ARCH}")
print(f"Encoder: {ENCODER_NAME}")
print(f"Tile resize (HxW): {TILE_RESIZE_HEIGHT}x{TILE_RESIZE_WIDTH}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Optuna trials: {N_TRIALS}")

### Model Training Utilities

In [ ]:
# CUDA device is selected in the Configuration cell (GPU_ID / DEVICE).

In [ ]:
import gc
import torch.nn as nn
from torch.optim import Adam, AdamW

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / CHECKPOINT_NAME
MODEL_DIR = PROJECT_ROOT / "models" / CHECKPOINT_NAME
TENSORBOARD_DIR = PROJECT_ROOT / "logs" / "tensorboard" / LOG_NAME

for d in [CHECKPOINT_DIR, MODEL_DIR, TENSORBOARD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_BUILDERS = {
    "DeepLabV3": smp.DeepLabV3,
    "Unet": smp.Unet,
    "UnetPlusPlus": smp.UnetPlusPlus,
}
if MODEL_ARCH not in MODEL_BUILDERS:
    raise ValueError(
        f"Unsupported MODEL_ARCH={MODEL_ARCH!r}. Choose one of: {', '.join(MODEL_BUILDERS)}"
    )


def freeze_encoder(model):
    if hasattr(model, "encoder"):
        for p in model.encoder.parameters():
            p.requires_grad = False


def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True


def load_checkpoint_forgiving(model, checkpoint_path: str, device=DEVICE):
    """Load a prior checkpoint as best-effort.

    - Accepts checkpoints saved as {"model_state_dict": ...}.
    - Only loads keys whose tensor shapes match.
    - Prevents crashes when NUM_CLASSES differs (segmentation head mismatch).
    """
    if not checkpoint_path:
        return

    ckpt_path = Path(checkpoint_path)
    if not ckpt_path.exists():
        raise FileNotFoundError(f"TRANSFER_CHECKPOINT_PATH not found: {ckpt_path}")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get("model_state_dict", ckpt)

    model_state = model.state_dict()
    filtered = {}
    loaded, skipped = 0, 0

    for k, v in state.items():
        if k in model_state and hasattr(v, "shape") and v.shape == model_state[k].shape:
            filtered[k] = v
            loaded += 1
        else:
            skipped += 1

    model.load_state_dict(filtered, strict=False)
    print(f"Loaded checkpoint (forgiving): {ckpt_path} | loaded={loaded} skipped={skipped}")


def build_model(
    num_classes,
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    load_transfer_checkpoint=True,
):
    model = MODEL_BUILDERS[MODEL_ARCH](
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=num_classes,
    )

    if TRANSFER_LEARNING and load_transfer_checkpoint:
        load_checkpoint_forgiving(model, TRANSFER_CHECKPOINT_PATH, device=DEVICE)

    return model.to(DEVICE)


def build_optimizer(params, name, lr, weight_decay):
    if name == "adamw":
        return AdamW(params, lr=lr, weight_decay=weight_decay)
    return Adam(params, lr=lr, weight_decay=weight_decay)


def build_scheduler(optimizer, name, num_epochs):
    """Build a LR scheduler by name. Returns None for 'none'."""
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    if name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, num_epochs // 3), gamma=0.5)
    return None


def compute_iou(pred, target, num_classes, exclude_background=True):
    """Mean IoU across classes. Excludes class 0 (background) by default."""
    start = 1 if exclude_background and num_classes > 1 else 0
    ious = []
    for cls in range(start, num_classes):
        pred_mask = (pred == cls)
        target_mask = (target == cls)
        intersection = (pred_mask & target_mask).sum().float()
        union = (pred_mask | target_mask).sum().float()
        iou = intersection / (union + 1e-6)
        ious.append(iou.item())
    return np.mean(ious) if ious else 0.0


def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    num_batches = 0

    for img_patch, mask_patch in train_loader:
        img_patch = img_patch.to(device)
        mask_patch = mask_patch.to(device)

        logits = model(img_patch)
        loss = criterion(logits, mask_patch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(1, num_batches)


def validate(model, val_loader, criterion, device, num_classes):
    model.eval()
    total_loss = 0.0
    total_iou = 0.0
    num_batches = 0

    with torch.no_grad():
        for img_patch, mask_patch in val_loader:
            img_patch = img_patch.to(device)
            mask_patch = mask_patch.to(device)

            logits = model(img_patch)
            loss = criterion(logits, mask_patch)
            pred = torch.argmax(logits, dim=1)
            iou = compute_iou(pred, mask_patch, num_classes, exclude_background=True)

            total_loss += loss.item()
            total_iou += iou
            num_batches += 1

    return total_loss / max(1, num_batches), total_iou / max(1, num_batches)


def run_training(config, trial=None):
    # Initialize all cleanup-relevant refs to None so finally can always reach them,
    # even if an exception is raised before they are assigned.
    model = None
    optimizer = None
    scheduler = None
    train_loader = None
    val_loader = None
    criterion = None
    writer = None

    # For Optuna trials: disable pin_memory and num_workers to prevent VRAM accumulation.
    # pin_memory allocates GPU buffers that don't always get freed between trials.
    trial_num_workers = 0 if trial is not None else NUM_WORKERS
    trial_pin_memory = False

    # Verify DEVICE is actually GPU
    if trial is not None and "cuda" not in str(DEVICE):
        print(f"WARNING: DEVICE={DEVICE} is not CUDA! This explains zero memory allocation!")

    try:
        train_loader, val_loader = create_dataloaders_full(
            data_root=DATA_ROOT,
            batch_size=config.get("batch_size", BATCH_SIZE),
            num_workers=trial_num_workers,
            image_height=config.get("image_height", IMAGE_HEIGHT),
            image_width=config.get("image_width", IMAGE_WIDTH),
            pin_memory=trial_pin_memory,
        )

        model = build_model(
            NUM_CLASSES,
            encoder_name=config.get("encoder", ENCODER_NAME),
        )

        # Print memory AFTER model creation (this is where memory should show up)
        if trial is not None:
            torch.cuda.synchronize()
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            print(f"[Trial {trial.number} AFTER MODEL] Allocated: {allocated:.2f}GB / Reserved: {reserved:.2f}GB")

        # Transfer learning schedule: optionally freeze encoder first.
        freeze_epochs = int(config.get("freeze_encoder_epochs", 0) or 0)
        unfreeze_after = bool(config.get("unfreeze_after_freeze", True))
        if config.get("transfer_learning") and freeze_epochs > 0:
            freeze_encoder(model)

        criterion = nn.CrossEntropyLoss()

        optimizer = build_optimizer(
            filter(lambda p: p.requires_grad, model.parameters()),
            name=config.get("optimizer", "adam"),
            lr=config["lr"],
            weight_decay=config["weight_decay"],
        )

        scheduler = build_scheduler(
            optimizer,
            name=config.get("scheduler", "none"),
            num_epochs=config["num_epochs"],
        )

        writer = SummaryWriter(log_dir=config["log_dir"]) if config.get("log_dir") else None

        history = {"train_loss": [], "val_loss": [], "val_iou": []}
        best_val_iou = -1.0
        best_epoch = -1
        patience = config.get("early_stop_patience")
        min_delta = float(config.get("early_stop_min_delta", 0.0))
        epochs_since_improve = 0

        for epoch in range(config["num_epochs"]):
            # Unfreeze after a short warmup period (transfer learning).
            if config.get("transfer_learning") and freeze_epochs > 0 and unfreeze_after and epoch == freeze_epochs:
                unfreeze_all(model)
                optimizer = build_optimizer(
                    model.parameters(),
                    name=config.get("optimizer", "adam"),
                    lr=float(config.get("lr_unfrozen", config["lr"])),
                    weight_decay=config["weight_decay"],
                )
                # Rebuild scheduler with remaining epochs after unfreeze.
                remaining = config["num_epochs"] - epoch
                scheduler = build_scheduler(optimizer, name=config.get("scheduler", "none"), num_epochs=remaining)

            train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
            val_loss, val_iou = validate(model, val_loader, criterion, DEVICE, NUM_CLASSES)

            if scheduler is not None:
                scheduler.step()

            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["val_iou"].append(val_iou)

            if config.get("verbose"):
                print(
                    f"Epoch {epoch+1}/{config['num_epochs']} | "
                    f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU (excl bg): {val_iou:.4f}"
                )

            if writer:
                writer.add_scalar("Loss/Train", train_loss, epoch)
                writer.add_scalar("Loss/Val", val_loss, epoch)
                writer.add_scalar("IoU/Val", val_iou, epoch)

            improved = val_iou > best_val_iou + min_delta
            if improved:
                best_val_iou = val_iou
                best_epoch = epoch + 1
                epochs_since_improve = 0
                if config.get("save_checkpoints"):
                    best_path = MODEL_DIR / "model_best.pt"
                    torch.save(
                        {
                            "epoch": epoch + 1,
                            "model_state_dict": model.state_dict(),
                            "arch": MODEL_ARCH,
                            "encoder": config.get("encoder", ENCODER_NAME),
                            "optimizer_state_dict": optimizer.state_dict(),
                            "train_loss": train_loss,
                            "val_loss": val_loss,
                            "val_iou": val_iou,
                        },
                        best_path,
                    )
                    if config.get("verbose"):
                        print(f"  -> Best checkpoint saved: {best_path}")
            else:
                epochs_since_improve += 1

            if trial is not None:
                trial.report(val_iou, epoch)
                if trial.should_prune():
                    # Raise immediately - finally block handles all cleanup.
                    raise optuna.exceptions.TrialPruned()

            if config.get("save_checkpoints") and ((epoch + 1) % config.get("save_interval", 5) == 0):
                ckpt_path = CHECKPOINT_DIR / f"model_epoch_{epoch+1}.pt"
                torch.save(
                    {
                        "epoch": epoch + 1,
                        "model_state_dict": model.state_dict(),
                        "arch": MODEL_ARCH,
                        "encoder": config.get("encoder", ENCODER_NAME),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "train_loss": train_loss,
                        "val_loss": val_loss,
                        "val_iou": val_iou,
                    },
                    ckpt_path,
                )
                if config.get("verbose"):
                    print(f"  -> Checkpoint saved: {ckpt_path}")

            if patience is not None and epochs_since_improve >= patience:
                if config.get("verbose"):
                    print(f"Early stopping at epoch {epoch+1} (best IoU={best_val_iou:.4f} at epoch {best_epoch})")
                break

        if config.get("save_checkpoints"):
            history_path = CHECKPOINT_DIR / "training_history.json"
            with open(history_path, "w") as f:
                json.dump(history, f, indent=2)
            if config.get("verbose"):
                print(f"\nTraining complete. History saved to {history_path}")

        return best_val_iou, history

    finally:
        # Runs on every exit path: normal return, TrialPruned, OOM RuntimeError, etc.
        # This prevents GPU memory from leaking across Optuna trials.
        if writer is not None:
            writer.close()
        # Delete ALL major objects to release GPU memory
        del model, optimizer, criterion
        if scheduler is not None:
            del scheduler
        del train_loader, val_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            # Aggressively clear CUDA memory caches (kernel caches can persist)
            for _ in range(5):
                torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            # Additional aggressive memory clearing
            torch.cuda.reset_accumulated_memory_stats()


### Optuna Hyperparameter Tuning

Search for strong validation IoU over batch size, learning rate, weight decay, and optimizer choices.

In [ ]:
print(f"=== TILE OPTUNA SETUP ===")
print(f"DEVICE: {DEVICE}")
print(f"BATCH_SIZE: {BATCH_SIZE}")
print(f"NUM_WORKERS: {NUM_WORKERS}")
print(f"MODEL_ARCH: {MODEL_ARCH}")
print(f"ENCODER_NAME: {ENCODER_NAME}")
print(f"TUNE_ENCODER: {TUNE_ENCODER}")
print(f"ENCODER_MODE: {ENCODER_MODE}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"Initial GPU Memory - Allocated: {allocated:.2f}GB / Reserved: {reserved:.2f}GB")
print("=" * 50)

optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna_db = Path(OPTUNA_DB_PATH)
optuna_db.parent.mkdir(parents=True, exist_ok=True)
OPTUNA_STORAGE = f"sqlite:///{optuna_db.resolve()}"
sampler = optuna.samplers.TPESampler(seed=42, multivariate=True)
pruner = optuna.pruners.HyperbandPruner(min_resource=5, max_resource=40, reduction_factor=3)


def _normalize_param_value(value):
    if isinstance(value, float):
        return float(f"{value:.12g}")
    return value


def _params_signature(params):
    return tuple(sorted((key, _normalize_param_value(value)) for key, value in params.items()))


def _choose_resume_study_name(storage_url, explicit_name=None):
    if explicit_name:
        return explicit_name
    summaries = optuna.study.get_all_study_summaries(storage=storage_url)
    prefix = f"{MODEL_ARCH}_tiled_optuna_{ENCODER_MODE}_"
    matching = [summary for summary in summaries if summary.study_name.startswith(prefix)]
    if not matching:
        return None
    return max(matching, key=lambda summary: summary.datetime_start or datetime.min).study_name


def print_gpu_memory(label=""):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
        print(f"[GPU Memory {label}] Allocated: {allocated:.2f}GB / Reserved: {reserved:.2f}GB / Total: {total:.2f}GB")


study = None
existing_param_signatures = set()


def objective(trial):
    trial_num = trial.number
    print_gpu_memory(f"Trial {trial_num} START")
    trial_encoder = (
        trial.suggest_categorical("encoder", ["resnet34", "resnet50"])
        if TUNE_ENCODER
        else ENCODER_NAME
    )
    config = {
        "lr": trial.suggest_float("lr", 1e-5, 3e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "optimizer": trial.suggest_categorical("optimizer", ["adam", "adamw"]),
        "scheduler": trial.suggest_categorical("scheduler", ["cosine", "step", "none"]),
        "batch_size": BATCH_SIZE,
        "encoder": trial_encoder,
        "num_epochs": TRIAL_EPOCHS,
        "early_stop_patience": 8,
        "early_stop_min_delta": 1e-3,
        "image_height": TILE_RESIZE_HEIGHT,
        "image_width": TILE_RESIZE_WIDTH,
        "save_checkpoints": False,
        "verbose": False,
    }
    signature_params = {
        key: config[key]
        for key in ("lr", "weight_decay", "optimizer", "scheduler", "batch_size")
    }
    if TUNE_ENCODER:
        signature_params["encoder"] = config["encoder"]
    trial_signature = _params_signature(signature_params)
    if trial_signature in existing_param_signatures:
        trial.set_user_attr("duplicate_combination", True)
        raise optuna.exceptions.TrialPruned("Duplicate hyperparameter combination from prior trials.")
    existing_param_signatures.add(trial_signature)
    best_val_iou, _ = run_training(config, trial=trial)
    del trial, config
    gc.collect()
    print_gpu_memory(f"Trial {trial_num} END")
    return best_val_iou


resume_name = _choose_resume_study_name(OPTUNA_STORAGE, explicit_name=RESUME_STUDY_NAME) if RESUME_PREVIOUS_STUDY else None
if resume_name:
    study = optuna.load_study(study_name=resume_name, storage=OPTUNA_STORAGE, sampler=sampler, pruner=pruner)
    print(f"Resuming existing study: {resume_name}")
else:
    study = optuna.create_study(direction="maximize", study_name=STUDY_NAME, storage=OPTUNA_STORAGE, load_if_exists=True, sampler=sampler, pruner=pruner)
    print(f"Created/loaded study: {study.study_name}")

for trial in study.trials:
    signature_params = dict(trial.params)
    if not TUNE_ENCODER:
        signature_params.pop("encoder", None)
    if signature_params:
        existing_param_signatures.add(_params_signature(signature_params))
print(f"Existing trials in study: {len(study.trials)}")
try:
    print(f"Current best before new trials: {study.best_trial.value:.6f}")
    print(f"Current best params: {study.best_trial.params}")
except ValueError:
    print("No completed trials yet in this study.")

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, gc_after_trial=True, catch=(RuntimeError, torch.cuda.OutOfMemoryError))
print_gpu_memory("Final")
try:
    print("Best trial:", study.best_trial.value, study.best_trial.params)
except ValueError:
    print("No completed trials are available yet in this study.")

### Train Final Model With Best Hyperparameters

Train DeepLabV3 with the selected trial and save best/periodic checkpoints.

In [ ]:
best_params = study.best_trial.params

final_config = {
    "batch_size": BATCH_SIZE,
    "lr": best_params["lr"],
    "weight_decay": best_params["weight_decay"],
    "optimizer": best_params["optimizer"],
    "scheduler": best_params.get("scheduler", "none"),
    "encoder": best_params.get("encoder", ENCODER_NAME),
    "num_epochs": NUM_EPOCHS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "early_stop_min_delta": 1e-4,
    "image_height": IMAGE_HEIGHT,
    "image_width": IMAGE_WIDTH,
    "save_checkpoints": True,
    "save_interval": 5,
    "log_dir": str(TENSORBOARD_DIR),
    "verbose": True,
}

best_val_iou, history = run_training(final_config)
print(f"Final training complete. Best Val IoU: {best_val_iou:.4f}")


In [ ]:
# Optional: transfer learning override for final training
# (Keeps Optuna tuning unchanged; only affects the final training run.)

if TRANSFER_LEARNING:
    final_config.update(
        {
            "transfer_learning": True,
            "freeze_encoder_epochs": FREEZE_ENCODER_EPOCHS,
            "unfreeze_after_freeze": UNFREEZE_AFTER_FREEZE,
            "lr_unfrozen": TRANSFER_LR_UNFROZEN,
        }
    )
    final_config["lr"] = TRANSFER_LR_FROZEN

    best_val_iou, history = run_training(final_config)
    print(f"Transfer-learning run complete. Best Val IoU: {best_val_iou:.4f}")


In [ ]:
# Required evaluation: metrics are computed per tile.
test_dataset = TiledSegmentationDataset(
    data_root=DATA_ROOT,
    split="test",
    transform=get_validation_augmentation_full(
        image_height=TILE_RESIZE_HEIGHT,
        image_width=TILE_RESIZE_WIDTH,
    ),
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
)

print(f"Loaded test dataset: {len(test_dataset)} tiles")

In [ ]:
# Load best model for evaluation using the architecture selected in MODEL_ARCH.
best_model_path = MODEL_DIR / "model_best.pt"
ckpt = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
EVAL_ENCODER = ckpt.get("encoder") or (final_config["encoder"] if "final_config" in globals() else ENCODER_NAME)

model = build_model(
    NUM_CLASSES,
    encoder_name=EVAL_ENCODER,
    encoder_weights=None,
    load_transfer_checkpoint=False,
)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print(f"Loaded best {MODEL_ARCH} model: {best_model_path} (encoder={EVAL_ENCODER})")
print(f"Checkpoint epoch={ckpt.get('epoch')} | val_iou={ckpt.get('val_iou'):.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import auc, roc_curve
from sklearn.metrics import ConfusionMatrixDisplay

# Confusion-matrix metrics are exact. ROC curves use a bounded, evenly spaced
# uint8 score sample so evaluation does not retain every probability map in RAM.
ROC_MAX_SAMPLES = 500_000
roc_samples_per_tile = max(1, ROC_MAX_SAMPLES // max(1, len(test_dataset)))
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
roc_targets = []
roc_probs = []

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        target_pixels = masks.numpy().reshape(-1).astype(np.int64, copy=False)
        pred_pixels = preds.cpu().numpy().reshape(-1).astype(np.int64, copy=False)
        cm += np.bincount(
            NUM_CLASSES * target_pixels + pred_pixels,
            minlength=NUM_CLASSES ** 2,
        ).reshape(NUM_CLASSES, NUM_CLASSES)

        probability_pixels = (probs * 255).to(torch.uint8).cpu().numpy().transpose(0, 2, 3, 1).reshape(-1, NUM_CLASSES)
        sample_count = min(roc_samples_per_tile, target_pixels.size)
        sample_indices = np.linspace(0, target_pixels.size - 1, num=sample_count, dtype=np.int64)
        roc_targets.append(target_pixels[sample_indices])
        roc_probs.append(probability_pixels[sample_indices])

roc_targets = np.concatenate(roc_targets)
roc_probs = np.concatenate(roc_probs)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

# Confusion matrix figure
disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
fig_cm, ax_cm = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax_cm, cmap="Blues", values_format=".1%", colorbar=True, xticks_rotation=45)
ax_cm.set_title("Confusion Matrix (Percent of True Class)")
fig_cm.tight_layout()

cm_png = METRICS_DIR / f"{METRICS_RUN_NAME}_confusion_matrix.png"
fig_cm.savefig(cm_png, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig_cm)

total_pixels = cm.sum()
acc = np.trace(cm) / total_pixels if total_pixels else 0.0
print(f"Accuracy: {acc:.4f}")

true_positives = np.diag(cm).astype(np.float64)
predicted_positives = cm.sum(axis=0)
actual_positives = cm.sum(axis=1)
precision = np.divide(true_positives, predicted_positives, out=np.zeros(NUM_CLASSES), where=predicted_positives != 0)
recall = np.divide(true_positives, actual_positives, out=np.zeros(NUM_CLASSES), where=actual_positives != 0)
f1_per_class = np.divide(2 * precision * recall, precision + recall, out=np.zeros(NUM_CLASSES), where=(precision + recall) != 0)
f1_macro = f1_per_class.mean()
print("F1 per class:")
for i, f1 in enumerate(f1_per_class):
    print(f"  {CLASS_NAMES[i]}: {f1:.4f}")
print(f"F1 Macro: {f1_macro:.4f}")

intersection = true_positives.astype(np.float32)
union = cm.sum(axis=0) + cm.sum(axis=1) - intersection
iou_per_class = np.divide(intersection, union, out=np.zeros_like(intersection), where=union != 0)
print("IoU per class:")
for i, iou in enumerate(iou_per_class):
    print(f"  {CLASS_NAMES[i]}: {iou:.4f}")

mean_iou_excl_bg = iou_per_class[1:].mean() if NUM_CLASSES > 1 else iou_per_class.mean()
print(f"Mean IoU (excl background): {mean_iou_excl_bg:.4f}")

# ROC figure from bounded uint8 samples.
fig_roc, ax_roc = plt.subplots(figsize=(7, 5))
for c in range(NUM_CLASSES):
    y_true_bin = (roc_targets == c).astype(int)
    y_score = roc_probs[:, c]

    if y_true_bin.sum() == 0:
        print(f"ROC skipped: {CLASS_NAMES[c]} (no positives in ROC sample)")
        continue

    fpr, tpr, _ = roc_curve(y_true_bin, y_score)
    roc_auc = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, label=f"{CLASS_NAMES[c]} (AUC={roc_auc:.3f})")

ax_roc.plot([0, 1], [0, 1], "k--")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.set_title(f"ROC Curves (One-vs-Rest, {len(roc_targets):,} sampled pixels)")
ax_roc.legend()
fig_roc.tight_layout()

roc_png = METRICS_DIR / f"{METRICS_RUN_NAME}_roc_curves.png"
fig_roc.savefig(roc_png, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig_roc)

print(f"Saved confusion matrix to: {cm_png}")
print(f"Saved ROC curves to: {roc_png}")

### Optional slow evaluation: stitched full boards

This cell averages native-resolution tile probability maps into each test board and reports board-level IoU/F1. It is intentionally separate from the required per-tile metrics and may be expensive for large boards.

In [ ]:
from collections import defaultdict
import torch.nn.functional as F
from sklearn.metrics import f1_score

with open(Path(DATA_ROOT) / "tile_manifest.json", encoding="utf-8") as handle:
    manifest = json.load(handle)
test_tile_ids = json.load(open(Path(DATA_ROOT) / "split.json", encoding="utf-8"))["test"]
by_board = defaultdict(list)
for tile_id in test_tile_ids:
    by_board[manifest[tile_id]["parent_board"]].append(tile_id)

board_scores = []
with torch.no_grad():
    for board_id, tile_ids in by_board.items():
        parent_height, parent_width = manifest[tile_ids[0]]["parent_size"]
        probability_sum = np.zeros((NUM_CLASSES, parent_height, parent_width), dtype=np.float32)
        counts = np.zeros((parent_height, parent_width), dtype=np.float32)
        for tile_id in tile_ids:
            ymin, ymax, xmin, xmax = manifest[tile_id]["box"]
            image = np.asarray(Image.open(Path(DATA_ROOT) / "images" / f"{tile_id}_Col.tif").convert("RGB"))
            transformed = get_validation_augmentation_full(
                image_height=TILE_RESIZE_HEIGHT, image_width=TILE_RESIZE_WIDTH
            )(image=image, mask=np.zeros(image.shape[:2], dtype=np.uint8))
            tile_input = transformed["image"].unsqueeze(0).to(DEVICE)
            tile_probs = torch.softmax(model(tile_input), dim=1)
            tile_probs = F.interpolate(
                tile_probs, size=(ymax - ymin, xmax - xmin), mode="bilinear", align_corners=False
            )[0].cpu().numpy()
            probability_sum[:, ymin:ymax, xmin:xmax] += tile_probs
            counts[ymin:ymax, xmin:xmax] += 1
        parent_mask = np.asarray(Image.open(Path(data_root) / "combined_masks" / f"{board_id}_mask.png"))
        stitched_pred = np.argmax(probability_sum / np.maximum(counts[None], 1), axis=0)
        valid = counts > 0
        board_scores.append({
            "board": board_id,
            "iou": np.mean([
                np.sum((stitched_pred[valid] == cls) & (parent_mask[valid] == cls)) /
                max(1, np.sum((stitched_pred[valid] == cls) | (parent_mask[valid] == cls)))
                for cls in range(1, NUM_CLASSES)
            ]),
            "f1": f1_score(parent_mask[valid].reshape(-1), stitched_pred[valid].reshape(-1), average="macro"),
        })
print("Stitched board metrics:", board_scores)
print("Mean stitched IoU:", np.mean([row["iou"] for row in board_scores]))
print("Mean stitched F1:", np.mean([row["f1"] for row in board_scores]))

In [ ]:
import torch.nn.functional as F
import torch.onnx
import torch.nn as nn
import numpy as np


class EBIExportWrapper(nn.Module):
    """Export raw uint8 tile RGB input and per-pixel class scores."""

    def __init__(self, seg_model, mean=None, std=None):
        super().__init__()
        self.seg_model = seg_model
        mean = [0.485, 0.456, 0.406] if mean is None else mean
        std = [0.229, 0.224, 0.225] if std is None else std
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1))

    def forward(self, x):
        x = x.float() / 255.0
        x = (x - self.mean) / self.std
        return F.softmax(self.seg_model(x), dim=1) * 100.0


onnx_dir = PROJECT_ROOT / "models" / "onnx"
onnx_dir.mkdir(parents=True, exist_ok=True)
onnx_path = onnx_dir / ONNX_NAME
ebi_model = EBIExportWrapper(model).to(DEVICE)
ebi_model.eval()
dummy_input = torch.randint(
    0, 256, (1, 3, TILE_RESIZE_HEIGHT, TILE_RESIZE_WIDTH), dtype=torch.uint8, device=DEVICE
)
torch.onnx.export(
    ebi_model,
    dummy_input,
    onnx_path.as_posix(),
    input_names=["input"],
    output_names=["output"],
    opset_version=12,
    dynamo=False,
    dynamic_axes={
        "input": {0: "batch", 2: "height", 3: "width"},
        "output": {0: "batch", 2: "height", 3: "width"},
    },
)
print(f"Exported ONNX tile model to: {onnx_path}")

import onnxruntime as ort
session = ort.InferenceSession(onnx_path.as_posix())
inp_meta = session.get_inputs()[0]
out_meta = session.get_outputs()[0]
test_input = np.random.randint(
    0, 256, (1, 3, TILE_RESIZE_HEIGHT, TILE_RESIZE_WIDTH)
).astype(np.uint8)
onnx_output = session.run(None, {inp_meta.name: test_input})[0]
print(f"ONNX Input: {inp_meta.name}, shape={inp_meta.shape}, type={inp_meta.type}")
print(f"ONNX Output: {out_meta.name}, shape={out_meta.shape}, type={out_meta.type}")
print(f"Class-sum range: {onnx_output.sum(axis=1).min():.2f}..{onnx_output.sum(axis=1).max():.2f}")
print("ONNX verification passed")

In [ ]:
from pathlib import Path
import sys
import json

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
notebooks_dir = project_root / "notebooks"
if str(notebooks_dir) not in sys.path:
    sys.path.append(str(notebooks_dir))

from functions.metrics_export import append_row_to_csv, build_run_row, first_available, guess_actual_epochs

local_vars = locals()
history = local_vars.get("history")
training_history_path = local_vars.get("history_path")
if training_history_path is None and "CHECKPOINT_DIR" in local_vars:
    training_history_path = local_vars["CHECKPOINT_DIR"] / "training_history.json"
if history is None and training_history_path is not None and Path(training_history_path).exists():
    with open(training_history_path, "r", encoding="utf-8") as f:
        history = json.load(f)

final_config = local_vars.get("final_config", {})
optuna_best_params = dict(local_vars["study"].best_trial.params) if "study" in local_vars else {}
with open(Path(DATA_ROOT) / "split.json", encoding="utf-8") as handle:
    tiled_split = json.load(handle)
train_split_size = len(tiled_split["train"])
val_split_size = len(tiled_split["val"])
test_split_size = len(tiled_split["test"])
with open(Path(DATA_ROOT) / "tile_manifest.json", encoding="utf-8") as handle:
    tile_manifest = json.load(handle)
board_counts = {
    split: len({tile_manifest[tile_id]["parent_board"] for tile_id in tiled_split[split]})
    for split in ("train", "val", "test")
}
tiles_per_board = tile_stats["tiles_per_board"]

extra_training_params = dict(final_config) if isinstance(final_config, dict) else {}
for key in ["ENCODER_NAME", "TRANSFER_LEARNING", "FREEZE_ENCODER_EPOCHS", "NUM_EPOCHS", "BATCH_SIZE", "NUM_WORKERS", "LEARNING_RATE", "OPTUNA_DB_PATH", "STUDY_NAME"]:
    if key in local_vars:
        extra_training_params[key] = local_vars[key]
extra_training_params.update({
    "TILE_MAX_X": TILE_MAX_X,
    "TILE_MAX_Y": TILE_MAX_Y,
    "TILE_STEP_X": TILE_STEP_X,
    "TILE_STEP_Y": TILE_STEP_Y,
    "TILE_EDGE_MODE": TILE_EDGE_MODE,
    "TILE_RESIZE_HEIGHT": TILE_RESIZE_HEIGHT,
    "TILE_RESIZE_WIDTH": TILE_RESIZE_WIDTH,
    "DROP_BACKGROUND_TRAIN_TILES": DROP_BACKGROUND_TRAIN_TILES,
    "MIN_TILE_DEFECT_PIXELS": MIN_TILE_DEFECT_PIXELS,
    "TILED_DATA_DIR": TILED_DATA_DIR,
    "tiles_train": train_split_size,
    "tiles_val": val_split_size,
    "tiles_test": test_split_size,
    "tiles_per_board_mean": tiles_per_board["mean"],
    "tiles_per_board_max": tiles_per_board["max"],
    "tile_manifest_path": str(Path(DATA_ROOT) / "tile_manifest.json"),
})

metrics_dir = Path(local_vars.get("METRICS_DIR", project_root / "data" / "metrics"))
row = build_run_row(
    notebook_name="Tiled_Optuna",
    model_name=str(first_available(local_vars, ["MODEL_ARCH", "CHECKPOINT_NAME"]) or "Unet"),
    raw_data_folder=first_available(local_vars, ["RAW_DIR"]),
    image_height=TILE_RESIZE_HEIGHT,
    image_width=TILE_RESIZE_WIDTH,
    batch_size=first_available(local_vars, ["BATCH_SIZE"]),
    num_workers=first_available(local_vars, ["NUM_WORKERS"]),
    num_epochs_config=first_available(local_vars, ["NUM_EPOCHS"]),
    num_epochs_actual=guess_actual_epochs(history),
    optimizer_name=final_config.get("optimizer") if isinstance(final_config, dict) else None,
    learning_rate=final_config.get("lr") if isinstance(final_config, dict) else None,
    weight_decay=final_config.get("weight_decay") if isinstance(final_config, dict) else None,
    loss_name="CrossEntropyLoss",
    num_classes=first_available(local_vars, ["NUM_CLASSES", "num_classes"]),
    class_names=first_available(local_vars, ["CLASS_NAMES", "class_names"]),
    train_split_size=train_split_size,
    val_split_size=val_split_size,
    test_split_size=test_split_size,
    accuracy=first_available(local_vars, ["acc", "accuracy", "pixel_acc"]),
    f1_macro=first_available(local_vars, ["f1_macro"]),
    f1_per_class=first_available(local_vars, ["f1_per_class"]),
    mean_iou=first_available(local_vars, ["mean_iou_excl_bg", "mean_iou"]),
    iou_per_class=first_available(local_vars, ["iou_per_class", "test_ious"]),
    checkpoint_path=first_available(local_vars, ["best_model_path", "deployment_checkpoint"]),
    training_history_path=training_history_path,
    onnx_path=first_available(local_vars, ["onnx_path"]),
    run_name=first_available(local_vars, ["METRICS_RUN_NAME", "CHECKPOINT_NAME"]),
    transfer_learning=first_available(local_vars, ["TRANSFER_LEARNING"]),
    transfer_checkpoint_path=first_available(local_vars, ["TRANSFER_CHECKPOINT_PATH"]),
    freeze_encoder_epochs=first_available(local_vars, ["FREEZE_ENCODER_EPOCHS"]),
    optuna_enabled=True,
    optuna_n_trials=first_available(local_vars, ["N_TRIALS"]),
    optuna_best_value=(local_vars["study"].best_trial.value if "study" in local_vars else None),
    optuna_best_params=optuna_best_params,
    extra_training_params=extra_training_params,
    extra_info={"boards_train": board_counts["train"], "boards_val": board_counts["val"], "boards_test": board_counts["test"]},
)

written_path = append_row_to_csv(metrics_dir / "TRAINING_RESULTS_SUMMARY.csv", row)
print(f"Appended metrics row to: {written_path}")